# EHR Timeline Transformer — Colab MEM + Regularized Fine-tune (Scheme 1)

Full pipeline on Colab with **Scheme 1** regularization experiments:

**prepare_data → MEM pretrain → reg_a + reg_bc fine-tune → compare → predictions**

## What Scheme 1 does
1. **reg_a**: config-only regularization (`configs/transformer_finetune_reg_a.yaml`)
2. **reg_bc**: full B+C regularization (`configs/transformer_finetune_reg.yaml`)
3. **compare**: `scripts/compare_reg_experiments.py` vs baseline / reg_a / reg_bc
4. **predict**: use the checkpoint with highest val mAP

## vs `colab_mem_train.ipynb`
| | `colab_mem_train.ipynb` | This notebook |
|---|------------------------|---------------|
| Fine-tune | baseline only | reg_a + reg_bc |
| Compare | none | automatic |
| Predictions | fixed `best_model.pt` | best of baseline / reg_a / reg_bc |

## Expected outputs
- `outputs/checkpoints/pretrain_best.pt`
- `outputs/checkpoints/best_model_reg_a.pt` + `val_metrics_reg_a.json`
- `outputs/checkpoints/best_model_reg.pt` + `val_metrics_reg.json`
- Comparison table + `outputs/predictions.csv`

**Note**: Baseline `best_model.pt` / `val_metrics.json` are **not** overwritten. If you have baseline metrics from `colab_mem_train.ipynb`, they will appear in the comparison table.

## Requirements
- **GPU runtime** required for pretrain and fine-tune.
- Raw `data/` on Google Drive (same layout as `colab_mem_train`).
- Remote branch must include regularization configs and scripts.

In [ ]:
# --- Edit these paths for your environment ---
REPO_URL = "https://github.com/ShutingXie/modeling_patient_timelines.git"
REPO_DIR = "modeling_patient_timelines"
REPO_BRANCH = "feature/mem-pretrain"

# Persistent locations on Google Drive (upload data once; reused every session)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive"
DRIVE_DATA_DIR = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_OUTPUTS_DIR = f"{DRIVE_PROJECT_DIR}/outputs"

# Optional: zip with raw data for first-time setup. Set to None after data is extracted.
# DATA_ZIP = f"{DRIVE_PROJECT_DIR}/ehr_data.zip"  # or None

# Wandb run names for this notebook session
WANDB_PRETRAIN_RUN = "colab_mem_pretrain_reg_nb_v1"
WANDB_REG_A_RUN = "colab_mem_finetune_reg_a_v1"
WANDB_REG_BC_RUN = "colab_mem_finetune_reg_v1"

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive", force_remount=True)
os.chdir(DRIVE_PROJECT_DIR)
print(os.getcwd())

In [ ]:
REQUIRED = [
    "patient_splits.csv",
    "target_conditions.csv",
    "test_anchors.csv",
    "train_val",
    "test",
]

In [ ]:
import zipfile
from pathlib import Path

data_dir = Path(DRIVE_DATA_DIR)
missing = [name for name in REQUIRED if not (data_dir / name).exists()]

if missing:
    if "DATA_ZIP" in globals() and DATA_ZIP and Path(DATA_ZIP).exists():
        print(f"First-time setup: extracting {DATA_ZIP}")
        data_dir.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP) as zf:
            names = zf.namelist()
            top_levels = {n.split("/")[0] for n in names if n.strip()}
            if top_levels == {"data"}:
                zf.extractall(data_dir.parent)
            else:
                data_dir.mkdir(parents=True, exist_ok=True)
                zf.extractall(data_dir)
        missing = [name for name in REQUIRED if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing data in {DRIVE_DATA_DIR}: {', '.join(missing)}. "
            "Upload the raw CSV folders to Drive or set DATA_ZIP."
        )
else:
    print(f"Data ready at {DRIVE_DATA_DIR}")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)

if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)
    print(f"Updated existing repo at {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    print(f"Cloned repo to {REPO_DIR}")

subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin"], check=True)
subprocess.run(
    ["git", "-C", str(repo_dir), "checkout", REPO_BRANCH],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repo_dir), "pull", "origin", REPO_BRANCH],
    check=True,
)

subprocess.run(
    ["pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")],
    check=True,
)

for link_name, target in [("data", DRIVE_DATA_DIR), ("outputs", DRIVE_OUTPUTS_DIR)]:
    link = repo_dir / link_name
    target_path = Path(target).resolve()
    target_path.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() and link.resolve() == target_path:
        continue
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        backup = repo_dir / f"{link_name}.colab_backup"
        if backup.exists():
            shutil.rmtree(backup)
        link.rename(backup)
        print(f"Backed up existing {link_name}/ to {backup.name}/")
    link.symlink_to(target_path, target_is_directory=True)
    print(f"Linked {link} -> {target_path}")

os.chdir(repo_dir)
print(f"Working directory: {repo_dir}")

In [ ]:
from pathlib import Path

for name in REQUIRED:
    path = Path("data") / name
    assert path.exists(), f"Missing data/{name}"

print("All required data files are present.")

## Data preparation

Build processed artifacts under `outputs/processed/`. Required before MEM pretrain so `vocab.json` includes the `[MASK]` token.

In [ ]:
!python scripts/prepare_data.py --config configs/transformer.yaml

In [ ]:
import json
from pathlib import Path

vocab_path = Path("outputs/processed/vocab.json")
assert vocab_path.exists(), "Run prepare_data first"

vocab = json.loads(vocab_path.read_text())
token_to_id = vocab["token_to_id"]
assert "[MASK]" in token_to_id, "vocab.json is missing [MASK]; re-run prepare_data after pulling MEM code"

processed = sorted(p.name for p in Path("outputs/processed").iterdir())
print(f"[MASK] token id: {token_to_id['[MASK]']}")
print("Processed artifacts:")
for name in processed:
    print(f"  - {name}")

In [ ]:
import torch

print(
    "CUDA available:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
)

## MEM pretrain

Pretrain uses only official **train** patients, split internally into `pretrain_train` / `pretrain_val`.

Outputs:
- `outputs/checkpoints/pretrain_best.pt`
- `outputs/metrics/pretrain_metrics.json`
- `outputs/plots/pretrain_curves.png`

**Skip this section** if `pretrain_best.pt` already exists on Drive and metrics look good.

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path("configs/transformer.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["wandb"]["run_name"] = WANDB_PRETRAIN_RUN
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f"Set wandb run_name to {cfg['wandb']['run_name']}")

In [ ]:
import wandb

wandb.login()

!python scripts/train_pretrain.py --config configs/transformer.yaml

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

metrics_path = Path("outputs/metrics/pretrain_metrics.json")
curves_path = Path("outputs/plots/pretrain_curves.png")
split_path = Path("outputs/processed/pretrain_split.json")
ckpt_path = Path("outputs/checkpoints/pretrain_best.pt")

for path in [metrics_path, curves_path, split_path, ckpt_path]:
    assert path.exists(), f"Missing {path}"

metrics = json.loads(metrics_path.read_text())
print("Pretrain metrics:", metrics)
display(Image(str(curves_path)))

split = json.loads(split_path.read_text())
print(
    f"pretrain_train={len(split['pretrain_train_ids'])}, "
    f"pretrain_val={len(split['pretrain_val_ids'])}"
)

## Scheme 1: Regularized fine-tune experiments

Runs **reg_a** then **reg_bc**, then compares against baseline (if `val_metrics.json` exists).

| Run | Config | Checkpoint | Metrics |
|-----|--------|------------|---------|
| reg_a | `transformer_finetune_reg_a.yaml` | `best_model_reg_a.pt` | `val_metrics_reg_a.json` |
| reg_bc | `transformer_finetune_reg.yaml` | `best_model_reg.pt` | `val_metrics_reg.json` |

Both use `outputs/checkpoints/pretrain_best.pt` and do **not** overwrite baseline `best_model.pt`.

### Option A: One-click (recommended)
Run the cell below to execute `scripts/run_finetune_reg_experiments.sh`.

### Option B: Step-by-step
If a run is interrupted, use the backup cells in the next section to resume from `reg_a`, `reg_bc`, or `compare` only.

In [ ]:
import wandb

wandb.login()

!bash scripts/run_finetune_reg_experiments.sh

### Backup: step-by-step Scheme 1 cells

Skip this section if the one-click cell above completed successfully.

In [ ]:
import yaml
import wandb
from pathlib import Path

cfg_path = Path("configs/transformer_finetune_reg_a.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["wandb"]["run_name"] = WANDB_REG_A_RUN
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f"Set wandb run_name to {cfg['wandb']['run_name']}")

wandb.login()

!python scripts/train_transformer.py \
    --config configs/transformer_finetune_reg_a.yaml \
    --pretrained-checkpoint outputs/checkpoints/pretrain_best.pt

In [ ]:
import yaml
import wandb
from pathlib import Path

cfg_path = Path("configs/transformer_finetune_reg.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["wandb"]["run_name"] = WANDB_REG_BC_RUN
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f"Set wandb run_name to {cfg['wandb']['run_name']}")

wandb.login()

!python scripts/train_transformer.py \
    --config configs/transformer_finetune_reg.yaml \
    --pretrained-checkpoint outputs/checkpoints/pretrain_best.pt

In [ ]:
!python scripts/compare_reg_experiments.py

## Compare results (detailed view)

Builds a summary table and reads checkpoint metadata. Training curves in `outputs/plots/training_curves.png` reflect only the **last** fine-tune run; use wandb for full curves.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

METRICS_DIR = Path("outputs/metrics")
CHECKPOINT_DIR = Path("outputs/checkpoints")

RUNS = {
    "baseline": {
        "metrics": METRICS_DIR / "val_metrics.json",
        "checkpoint": CHECKPOINT_DIR / "best_model.pt",
        "config": "configs/transformer.yaml",
    },
    "reg_a": {
        "metrics": METRICS_DIR / "val_metrics_reg_a.json",
        "checkpoint": CHECKPOINT_DIR / "best_model_reg_a.pt",
        "config": "configs/transformer_finetune_reg_a.yaml",
    },
    "reg_bc": {
        "metrics": METRICS_DIR / "val_metrics_reg.json",
        "checkpoint": CHECKPOINT_DIR / "best_model_reg.pt",
        "config": "configs/transformer_finetune_reg.yaml",
    },
}

rows = []
for run_name, paths in RUNS.items():
    row = {"run": run_name, "config": paths["config"], "status": "missing"}
    if paths["metrics"].exists():
        metrics = json.loads(paths["metrics"].read_text())
        row.update(
            {
                "status": "ok",
                "macro_auroc": metrics.get("macro_auroc"),
                "mAP": metrics.get("mAP"),
                "val_loss": metrics.get("loss"),
            }
        )
    if paths["checkpoint"].exists():
        ckpt = torch.load(paths["checkpoint"], map_location="cpu", weights_only=False)
        row["best_epoch"] = ckpt.get("epoch")
        row["regularization"] = str(ckpt.get("regularization"))
    rows.append(row)

summary = pd.DataFrame(rows)
display(summary)

valid = summary[summary["status"] == "ok"]
if valid.empty:
    raise RuntimeError("No fine-tune metrics found. Run Scheme 1 experiments first.")

best_idx = valid["mAP"].astype(float).idxmax()
best_run = valid.loc[best_idx, "run"]
best_map = float(valid.loc[best_idx, "mAP"])
print(f"\nRecommended run: {best_run} (val mAP={best_map:.4f})")
print(f"Checkpoint: {RUNS[best_run]['checkpoint']}")

## Test predictions (best checkpoint)

Generate `predictions.csv` using the run with highest validation mAP from the comparison above.

In [ ]:
best_ckpt = RUNS[best_run]["checkpoint"]
assert best_ckpt.exists(), f"Missing checkpoint: {best_ckpt}"

print(f"Using checkpoint from run={best_run}: {best_ckpt}")

!python scripts/make_predictions.py \
    --checkpoint {str(best_ckpt)} \
    --output outputs/predictions.csv

In [ ]:
from google.colab import files

files.download("outputs/predictions.csv")